# Librerías

In [1]:
import argparse
import json
import time
from pathlib import Path

import requests

Usage policy [from here](https://wiki.openstreetmap.org/wiki/Overpass_API): Max safe usage: <10.000 queries/day and <1Gb/day

In [74]:
# Overpass API endpoint (use mirror if rate-limited)
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OVERPASS_MIRROR = "https://overpass.kumi.systems/api/interpreter"

QUERY_TIME_OUT = 600  # seconds

In [104]:
def build_query(city: str, country: str, bbox: tuple[float, float, float, float], time_out: int=600) -> str:
    """Insert bounding box into Overpass query template."""
    south, west, north, east = bbox
    return f"""
            [out:json][timeout:{time_out}];
            (
                // Query all highways within {city},{country}
                way["highway"~"^(motorway|trunk|primary|secondary|tertiary|unclassified|residential)$"]
                ({south},{west},{north},{east});
            );
            out body geom;
            """

In [105]:
def run_query(query: str, time_out: int = 600, retries: int = 3) -> dict:
    """POST query to Overpass API with retry logic."""
    for attempt in range(retries):
        try:
            print(f"[query] Sending Overpass query (attempt {attempt + 1}/{retries})...")
            response = requests.post(
                OVERPASS_URL,
                data={"data": query},
                timeout=time_out,
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.Timeout:
            print(f"[query] Timeout — retrying in 10s...")
            time.sleep(10)
        except requests.exceptions.HTTPError as e:
            if response.status_code == 400:
                print(f"[query] Bad request — check your query syntax.")
                raise e
            elif response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 60))
                print(f"[query] Rate limited — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise e

    raise RuntimeError("Overpass query failed after all retries.")

In [106]:
def overpass_to_geojson(data: dict) -> dict:
    """
    Convert Overpass JSON response to GeoJSON FeatureCollection.
    Each OSM way becomes a LineString Feature with all tags as properties.
    """
    features = []

    for element in data.get("elements", []):
        # Security check
        if element["type"] != "way":
            continue

        # Build geometry from node coordinates
        coords = [
            [node["lon"], node["lat"]]
            for node in element.get("geometry", [])
        ]

        # Skip features without valid geometry (a node or an element with missing geometry)
        if len(coords) < 2:
            continue

        tags = element.get("tags", {})

        feature = {
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": coords,
            },
            "properties": {
                "osm_id": element["id"],
                "highway": tags.get("highway"),
                "name": tags.get("name"),
                "name_es": tags.get("name:es"),
                # Lane-related tags
                "lanes": tags.get("lanes"),
                "lanes_forward": tags.get("lanes:forward"),
                "lanes_backward": tags.get("lanes:backward"),
                "turn_lanes": tags.get("turn:lanes"),
                # Road characteristics
                "oneway": tags.get("oneway"),
                "maxspeed": tags.get("maxspeed"),
                "surface": tags.get("surface"),
                "width": tags.get("width"),
                # Raw tags for reference
                "_all_tags": json.dumps(tags, ensure_ascii=False),
            },
        }
        features.append(feature)

    geojson = {
        "type": "FeatureCollection",
        "features": features,
    }
    return geojson

In [107]:
# Build query
# City's bounding box (extreme points defining a circumscribed rectangle): 
# south, west, north, east
# CITY_BBOX = (4.46, -74.22, 4.84, -73.99) # Bogotá's bounding box
CITY_BBOX = (4.64, -74.09, 4.69, -74.06) # Localidad de  Barrios Unidos
CITY_NAME = "Bogotá"
CITY_COUNTRY = "Colombia"

In [108]:
query=build_query(CITY_NAME, CITY_COUNTRY, CITY_BBOX, QUERY_TIME_OUT)

In [109]:
fetched_data = run_query(query, QUERY_TIME_OUT)

[query] Sending Overpass query (attempt 1/3)...


In [110]:
total = len([e for e in fetched_data["elements"] if e["type"] == "way"])
print(f"[query] Retrieved {total} road segments from OSM")
lanes_geojson = overpass_to_geojson(fetched_data)
with_lanes = sum(1 for f in lanes_geojson["features"] if f["properties"]["lanes"])
print(f"[query] {with_lanes}/{total} segments have 'lanes' tag "
        f"({100 * with_lanes / total:.1f}%)")

[query] Retrieved 2790 road segments from OSM
[query] 2508/2790 segments have 'lanes' tag (89.9%)


In [111]:
with open("../results/lanes_sample.geojson", "w", encoding="utf-8") as f:
        json.dump(lanes_geojson, f, ensure_ascii=False, indent=2)

In [112]:
import folium

# ── Load ──────────────────────────────────────────────────────────────────────
# with open("lanes_sample.geojson", encoding="utf-8") as f:
#     features = json.load(f)  # bare array, not a FeatureCollection

# # Wrap into a proper FeatureCollection for Folium
geojson = {"type": "FeatureCollection", "features": lanes_geojson["features"][:20]}

# ── Color by lane count ───────────────────────────────────────────────────────
LANE_COLORS = {"1": "#2ecc71", "2": "#3498db", "3": "#f39c12", "4": "#e74c3c"}

def style_fn(feature):
    lanes = str(feature["properties"].get("lanes", "1"))
    return {
        "color": LANE_COLORS.get(lanes, "#95a5a6"),
        "weight": int(lanes) + 1,   # thicker line = more lanes
        "opacity": 0.85,
    }

def highlight_fn(feature):
    return {"color": "white", "weight": 5, "opacity": 1}

# ── Build map ─────────────────────────────────────────────────────────────────
m = folium.Map(location=[4.666, -74.068], zoom_start=15, tiles="CartoDB dark_matter")

folium.GeoJson(
    geojson,
    style_function=style_fn,
    highlight_function=highlight_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "highway", "lanes", "oneway", "maxspeed"],
        aliases=["Street", "Type", "Lanes", "Oneway", "Max speed"],
        localize=True,
    ),
).add_to(m)

# ── Legend ────────────────────────────────────────────────────────────────────
legend = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
            background:white;padding:12px;border-radius:8px;font-family:Arial;font-size:13px">
  <b>Lanes</b><br>
  <span style="background:#2ecc71;padding:2px 10px;border-radius:3px">&nbsp;</span> 1 lane<br>
  <span style="background:#3498db;padding:2px 10px;border-radius:3px">&nbsp;</span> 2 lanes<br>
  <span style="background:#f39c12;padding:2px 10px;border-radius:3px">&nbsp;</span> 3 lanes<br>
  <span style="background:#e74c3c;padding:2px 10px;border-radius:3px">&nbsp;</span> 4 lanes<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend))

m  # displays inline in Jupyter